In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/clean_orders.csv', parse_dates=['order_date', 'ship_date', 'order_month'])
print(df.shape)

(172765, 20)


In [3]:
def scorecard(g):
    return pd.Series({
        'orders': g['order_id'].nunique(),
        'sales_$': g['sales'].sum(),
        'late_%': g['is_late'].mean() * 100,
        'avg_promised_days': g['days_scheduled'].mean(),
        'avg_actual_days': g['days_real'].mean(),
        'std_actual_days': g['days_real'].std(),
        'avg_delay_when_late': g.loc[g['is_late'] == 1, 'delay_days'].mean(),
        'on_time_%': (1 - g['is_late'].mean()) * 100
    })

mode_sc = df.groupby('ship_mode').apply(scorecard).round(2)
mode_sc.sort_values('late_%', ascending=False)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_18204\806625212.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mode_sc = df.groupby('ship_mode').apply(scorecard).round(2)


,orders,sales_$,late_%,avg_promised_days,avg_actual_days,std_actual_days,avg_delay_when_late,on_time_%
ship_mode,,,,,,,,
First Class,9602.0,5408068.56,100.00,1.0,2.00,0.00,1.00,0.00
Second Class,12256.0,6860661.56,79.83,2.0,3.99,1.41,2.50,20.17
Same Day,3407.0,1854872.21,47.93,0.0,0.48,0.50,1.00,52.07
Standard Class,37632.0,21090827.32,39.77,4.0,3.99,1.42,1.51,60.23


In [4]:
region_sc = df.groupby('region').apply(scorecard).round(2)
print(region_sc.sort_values('late_%', ascending=False)[['orders', 'sales_$', 'late_%', 'avg_actual_days', 'std_actual_days']])
print()
print("Spread between best and worst region (late %):",
      round(region_sc['late_%'].max() - region_sc['late_%'].min(), 1), "points")

                 orders     sales_$  late_%  avg_actual_days  std_actual_days
region                                                                       
Central Africa    533.0   315769.36   60.15             3.56             1.67
Western Europe   9563.0  5628938.55   58.53             3.50             1.63
South of  USA    1280.0   748217.15   58.52             3.50             1.64
South Asia       3219.0  1498811.04   58.35             3.50             1.61
East Africa       589.0   363376.45   58.17             3.51             1.62
East of USA      2219.0  1313179.39   58.17             3.50             1.65
Southeast Asia   4173.0  1850920.76   57.98             3.50             1.62
West Asia        1935.0  1122755.80   57.81             3.48             1.60
Eastern Europe   1246.0   747370.89   57.65             3.50             1.65
US Center        1852.0  1107813.81   57.53             3.46             1.62
South America    4738.0  2812832.09   57.18             3.51    

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_18204\1102514124.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  region_sc = df.groupby('region').apply(scorecard).round(2)


In [5]:
pivot = (df.pivot_table(index='region', columns='ship_mode', values='is_late', aggfunc='mean') * 100).round(1)
print(pivot)
print()
print("Range across regions within each mode (points):")
print((pivot.max() - pivot.min()).round(1))

ship_mode        First Class  Same Day  Second Class  Standard Class
region                                                              
Canada                 100.0      24.1          81.2            30.0
Caribbean              100.0      52.3          82.5            36.8
Central Africa         100.0      30.4          82.8            44.3
Central America        100.0      48.0          78.3            40.1
Central Asia           100.0      20.3          90.2            42.6
East Africa            100.0      43.2          79.0            41.6
East of USA            100.0      50.4          78.2            41.6
Eastern Asia           100.0      49.4          83.2            39.0
Eastern Europe         100.0      36.4          76.6            40.9
North Africa           100.0      58.4          81.9            38.0
Northern Europe        100.0      41.0          79.2            39.5
Oceania                100.0      43.0          79.5            40.5
South America          100.0      

In [6]:
print((df.groupby('segment')['is_late'].mean() * 100).round(1), '\n')
print((df.groupby('category')['is_late'].mean() * 100).sort_values(ascending=False).head(5).round(1), '\n')
print((df.groupby(df['order_date'].dt.month)['is_late'].mean() * 100).round(1))

segment
Consumer       57.3
Corporate      57.1
Home Office    57.6
Name: is_late, dtype: float64 

category
Golf Bags & Carts    68.9
Lacrosse             62.6
Pet Supplies         61.4
Cameras              61.2
Accessories          59.8
Name: is_late, dtype: float64 

order_date
1     56.8
2     57.2
3     57.5
4     56.7
5     57.7
6     57.3
7     56.7
8     58.0
9     57.8
10    56.8
11    57.1
12    57.7
Name: is_late, dtype: float64


In [7]:
mix = (df['ship_mode'].value_counts(normalize=True) * 100).round(1)
print(mix, '\n')

new_promise = {'First Class': 2, 'Second Class': 4, 'Standard Class': 4, 'Same Day': 1}
df['promise_new'] = df['ship_mode'].map(new_promise)
df['late_new'] = (df['days_real'] > df['promise_new']).astype(int)

out = pd.DataFrame({'current_late_%': df.groupby('ship_mode')['is_late'].mean() * 100,
                    'proposed_late_%': df.groupby('ship_mode')['late_new'].mean() * 100}).round(1)
print(out)
print("\nOverall late %:", round(df['is_late'].mean() * 100, 1), "->", round(df['late_new'].mean() * 100, 1))

ship_mode
Standard Class    59.7
Second Class      19.6
First Class       15.3
Same Day           5.4
Name: proportion, dtype: float64 

                current_late_%  proposed_late_%
ship_mode                                      
First Class              100.0              0.0
Same Day                  47.9              0.0
Second Class              79.8             39.8
Standard Class            39.8             39.8

Overall late %: 57.3 -> 31.5


In [ ]:
mode_sc.to_csv('../data/scorecard_mode.csv')
region_sc.to_csv('../data/scorecard_region.csv')
pivot.to_csv('../data/late_by_region_mode.csv')